<a href="https://colab.research.google.com/github/Sai875/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit

Audit the anonymized sample for useful signals, missingness, and measurement caveats before building a baseline.

Run the cells from top to bottom. The source dataset is the anonymized public starter sample; do not add private client data or credentials to this notebook.


## 1. Load and inspect

In [ ]:
import pandas as pd
import numpy as np

DATA_URL = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_URL)
print("Shape:", df.shape)
print("Column names:", df.columns.tolist())
display(df.dtypes.rename("dtype").to_frame())

## 2. Missingness and target balance

In [ ]:
missing = (df.isna().mean().mul(100).sort_values(ascending=False).rename("missing_pct").to_frame())
display(missing.head(15).round(2))
if "is_declining_label" in df:
    target = pd.to_numeric(df["is_declining_label"], errors="coerce")
elif "trend_direction" in df:
    target = df["trend_direction"].astype("string").str.lower().eq("down").astype(int)
else:
    raise KeyError("No recognized decline proxy column found.")
print("Observed decline-proxy rate:", round(float(target.mean()), 4))

## 3. Compare the proxy across interpretable groups

In [ ]:
for col in ["position_tier", "staleness_bucket", "ga4_data_available"]:
    if col not in df.columns:
        continue
    temp = df[[col]].copy()
    temp["decline_proxy"] = target
    summary = temp.groupby(col, dropna=False)["decline_proxy"].agg(rows="size", decline_rate="mean").sort_values("rows", ascending=False)
    print("\\nGroup:", col)
    display(summary.round(3))

## 4. Write an evidence-based audit summary

In [ ]:
numeric = df.select_dtypes(include="number")
if len(numeric.columns) >= 2:
    corr = numeric.corr(numeric_only=True)
    if "is_declining_label" in corr.columns:
        signal = corr["is_declining_label"].drop("is_declining_label", errors="ignore").sort_values(key=lambda s: s.abs(), ascending=False)
        display(signal.head(10).to_frame("correlation_with_label").round(3))
    else:
        print("No numeric is_declining_label column; do not interpret correlations as causal evidence.")
print("Audit complete: check missingness, subgroup size, label construction, and whether each field is available at decision time.")